In [7]:
import os
from google.colab import drive
drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/HPC')

Mounted at /content/drive


In [8]:
!uname -a && cat /etc/*release

Linux c51b66b47cad 6.6.122+ #1 SMP Thu Apr 30 18:17:14 UTC 2026 x86_64 x86_64 x86_64 GNU/Linux
DISTRIB_ID=Ubuntu
DISTRIB_RELEASE=22.04
DISTRIB_CODENAME=jammy
DISTRIB_DESCRIPTION="Ubuntu 22.04.5 LTS"
PRETTY_NAME="Ubuntu 22.04.5 LTS"
NAME="Ubuntu"
VERSION_ID="22.04"
VERSION="22.04.5 LTS (Jammy Jellyfish)"
VERSION_CODENAME=jammy
ID=ubuntu
ID_LIKE=debian
HOME_URL="https://www.ubuntu.com/"
SUPPORT_URL="https://help.ubuntu.com/"
BUG_REPORT_URL="https://bugs.launchpad.net/ubuntu/"
PRIVACY_POLICY_URL="https://www.ubuntu.com/legal/terms-and-policies/privacy-policy"
UBUNTU_CODENAME=jammy


In [9]:
!pwd
!ls -la

/content/drive/MyDrive/HPC
total 1010
-rw------- 1 root root    1565 Jun 27 14:20 matmul.c
-rw------- 1 root root    4382 Jun 27 14:20 matmul_cuda.cu
-rw------- 1 root root 1010440 Jun 27 14:20 matmul_gpu
-rw------- 1 root root   16464 Jun 27 14:20 matmul_seq


In [10]:
!lscpu

Architecture:                x86_64
  CPU op-mode(s):            32-bit, 64-bit
  Address sizes:             46 bits physical, 48 bits virtual
  Byte Order:                Little Endian
CPU(s):                      2
  On-line CPU(s) list:       0,1
Vendor ID:                   GenuineIntel
  Model name:                Intel(R) Xeon(R) CPU @ 2.00GHz
    CPU family:              6
    Model:                   85
    Thread(s) per core:      2
    Core(s) per socket:      1
    Socket(s):               1
    Stepping:                3
    BogoMIPS:                4000.35
    Flags:                   fpu vme de pse tsc msr pae mce cx8 apic sep mtrr pg
                             e mca cmov pat pse36 clflush mmx fxsr sse sse2 ss h
                             t syscall nx pdpe1gb rdtscp lm constant_tsc rep_goo
                             d nopl xtopology nonstop_tsc cpuid tsc_known_freq p
                             ni pclmulqdq ssse3 fma cx16 pcid sse4_1 sse4_2 x2ap
                   

In [11]:
!nvidia-smi

Sat Jun 27 14:42:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             15W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [12]:
%%writefile matmul.c
#include <stdio.h>
#include <stdlib.h>
#include <omp.h>

int main(int argc, char **argv) {
    if (argc < 2) {
        printf("Error: specify the dimension N.\n");
        printf("Usage: %s <N>\n", argv[0]);
        return 1;
    }

    int n = atoi(argv[1]);
    int i, j, k;

    double (*a)[n] = malloc(sizeof(double[n][n]));
    double (*b)[n] = malloc(sizeof(double[n][n]));
    double (*c)[n] = malloc(sizeof(double[n][n]));

    if (!a || !b || !c) {
        printf("Memory allocation error for N=%d\n", n);
        return 1;
    }
    for (i = 0; i < n; i++) {
        for (j = 0; j < n; j++) {
            a[i][j] = 2.0;
            b[i][j] = 3.0;
            c[i][j] = 0.0;
        }
    }

    double start_time = omp_get_wtime();

    for (i = 0; i < n; ++i) {
        for (k = 0; k < n; k++) {
            for (j = 0; j < n; ++j) {
                c[i][j] += a[i][k] * b[k][j];
            }
        }
    }

    double run_time = omp_get_wtime() - start_time;
    printf("Computation time (N=%d): %f seconds\n", n, run_time);

    #ifdef DEBUG
    FILE *f = fopen("mat-res.txt", "w");
    if (!f) {
        perror("fopen error");
        return 1;
    }

    fprintf(f, "%d\n\n", n);

    int print_limit = (n < 1000) ? n : 1000;

    for (int i = 0; i < print_limit; i++) {
        for (int j = 0; j < print_limit; j++) {
            fprintf(f, "%.0f ", c[i][j]);
        }
        fprintf(f, "\n");
    }
    fclose(f);
    printf("Debug file 'mat-res.txt' successfully saved.\n");
    #endif
    free(a);
    free(b);
    free(c);

    return 0;
}

Overwriting matmul.c


In [13]:
!gcc -O3 -fopenmp matmul.c -o matmul_seq



In [14]:
!time ./matmul_seq 5000

Computation time (N=5000): 91.537347 seconds

real	1m31.851s
user	1m30.493s
sys	0m0.364s


In [15]:
%%writefile matmul_cuda.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>

// ---------------------------------------------------------
// GPU KERNEL: Executed on the graphics card
// ---------------------------------------------------------
__global__ void matMulKernel(double *a, double *b, double *c, int n) {
    // 1. Calculate the global 2D coordinates of the thread
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    int row = blockIdx.y * blockDim.y + threadIdx.y;

    // 2. Check that the thread is within the matrix boundaries
    if (row < n && col < n) {
        double sum = 0.0;

        // 3. The actual computation (The only remaining loop!)
        // Notice how we access the 1D array simulating a 2D matrix (row * n + col)
        for (int k = 0; k < n; k++) {
            sum += a[row * n + k] * b[k * n + col];
        }

        // 4. Save the result
        c[row * n + col] = sum;
    }
}

// ---------------------------------------------------------
// MAIN HOST: Executed on the CPU
// ---------------------------------------------------------
int main(int argc, char **argv) {
    if (argc < 2) {
        printf("Error: specify the dimension N.\n");
        return 1;
    }
    int n = atoi(argv[1]);
    size_t size = (size_t)n * n * sizeof(double);

    // --- HOST MEMORY ALLOCATION (CPU) ---
    // We use "flattened" 1D arrays because they are the safest for GPU transfers
    double *h_a = (double *)malloc(size);
    double *h_b = (double *)malloc(size);
    double *h_c = (double *)malloc(size);

    if (!h_a || !h_b || !h_c) {
        printf("Host RAM allocation error for N=%d\n", n);
        return 1;
    }

    // Data initialization on the CPU
    for (int i = 0; i < n * n; i++) {
        h_a[i] = 2.0;
        h_b[i] = 3.0;
        h_c[i] = 0.0;
    }

    // --- PREPARATION OF CUDA TIMERS ---
    cudaEvent_t start, stop_h2d, stop_kernel, stop_d2h;
    cudaEventCreate(&start);
    cudaEventCreate(&stop_h2d);
    cudaEventCreate(&stop_kernel);
    cudaEventCreate(&stop_d2h);

    // --- DEVICE MEMORY ALLOCATION (GPU) ---
    double *d_a, *d_b, *d_c;
    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    // =======================================================
    // 1. DATA TRANSFER: CPU -> GPU (HostToDevice)
    // =======================================================
    cudaEventRecord(start); // Start the timer

    cudaMemcpy(d_a, h_a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, h_b, size, cudaMemcpyHostToDevice);

    cudaEventRecord(stop_h2d); // Record end of H2D

    // =======================================================
    // 2. KERNEL EXECUTION
    // =======================================================
    // Define the magic block in multiples of 32 (256 total threads)
    dim3 threadsPerBlock(16, 16);
    // Calculate the grid by rounding up to cover the entire matrix
    dim3 numBlocks((n + 15) / 16, (n + 15) / 16);

    matMulKernel<<<numBlocks, threadsPerBlock>>>(d_a, d_b, d_c, n);

    cudaEventRecord(stop_kernel); // Record end of GPU computation

    // =======================================================
    // 3. RESULT TRANSFER: GPU -> CPU (DeviceToHost)
    // =======================================================
    cudaMemcpy(h_c, d_c, size, cudaMemcpyDeviceToHost);

    cudaEventRecord(stop_d2h); // Record end of D2H
    cudaEventSynchronize(stop_d2h); // Wait for everything to completely finish

    // --- CALCULATION AND PRINTING OF TIMES ---
    float ms_h2d, ms_kernel, ms_d2h;
    cudaEventElapsedTime(&ms_h2d, start, stop_h2d);
    cudaEventElapsedTime(&ms_kernel, stop_h2d, stop_kernel);
    cudaEventElapsedTime(&ms_d2h, stop_kernel, stop_d2h);

    printf("Matrix Dimension N           : %d\n", n);
    printf("Transfer Time CPU -> GPU     : %f ms\n", ms_h2d);
    printf("GPU Kernel Computation Time  : %f ms\n", ms_kernel);
    printf("Transfer Time GPU -> CPU     : %f ms\n", ms_d2h);
    printf("Total Time (Data + Compute)  : %f ms\n", ms_h2d + ms_kernel + ms_d2h);

    // --- MEMORY CLEANUP ---
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    free(h_a); free(h_b); free(h_c);

    // Destroy the timers
    cudaEventDestroy(start); cudaEventDestroy(stop_h2d);
    cudaEventDestroy(stop_kernel); cudaEventDestroy(stop_d2h);

    return 0;
}

Overwriting matmul_cuda.cu


In [16]:
!nvcc -arch=sm_75 -O3 matmul_cuda.cu -o matmul_gpu

In [17]:
!find / -type f -name nsys 2>/dev/null

/opt/nvidia/nsight-compute/2025.1.1/host/target-linux-x64/nsys


In [18]:
!/content/matmul_gpu 5000

Matrix Dimension N           : 5000
Transfer Time CPU -> GPU     : 83.434525 ms
GPU Kernel Computation Time  : 1213.183350 ms
Transfer Time GPU -> CPU     : 42.291744 ms
Total Time (Data + Compute)  : 1338.909668 ms


In [19]:
!nvprof /content/matmul_gpu 5000

==2087== NVPROF is profiling process 2087, command: /content/matmul_gpu 5000
Matrix Dimension N           : 5000
Transfer Time CPU -> GPU     : 91.693245 ms
GPU Kernel Computation Time  : 1168.375854 ms
Transfer Time GPU -> CPU     : 45.116322 ms
Total Time (Data + Compute)  : 1305.185425 ms
==2087== Profiling application: /content/matmul_gpu 5000
==2087== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   89.58%  1.16818s         1  1.16818s  1.16818s  1.16818s  matMulKernel(double*, double*, double*, int)
                    6.99%  91.201ms         2  45.601ms  44.735ms  46.466ms  [CUDA memcpy HtoD]
                    3.43%  44.742ms         1  44.742ms  44.742ms  44.742ms  [CUDA memcpy DtoH]
      API calls:   90.02%  1.30489s         3  434.96ms  44.953ms  1.21326s  cudaMemcpy
                    9.61%  139.38ms         4  34.844ms     883ns  139.37ms  cudaEventCreate
                    0.26%  3.7347ms         3 

In [20]:
!/opt/nvidia/nsight-compute/2025.1.1/host/target-linux-x64/nsys profile -t cuda,osrt -o /content/drive/MyDrive/HPC/report_matmul --stats=true --force-overwrite=true ./matmul_gpu 5000

Invalid plugin configuration: Executable path does not exist: /opt/nvidia/nsight-compute/2025.1.1/host/target-linux-x64/plugins/efa_metrics/nic_sampler
Matrix Dimension N           : 5000
Transfer Time CPU -> GPU     : 85.764702 ms
GPU Kernel Computation Time  : 1213.798950 ms
Transfer Time GPU -> CPU     : 42.091393 ms
Total Time (Data + Compute)  : 1341.655029 ms
Generating '/tmp/nsys-report-e706.qdstrm'
[1/7] [========================100%] report_matmul.nsys-rep
[2/7] [========================100%] report_matmul.sqlite
[3/7] Executing 'osrt_sum' stats report

 Time (%)  Total Time (ns)  Num Calls    Avg (ns)      Med (ns)     Min (ns)   Max (ns)    StdDev (ns)            Name         
 --------  ---------------  ---------  ------------  -------------  --------  -----------  ------------  ----------------------
     91.3    1,790,436,176         27  66,312,451.0  100,117,704.0     1,797  287,409,178  62,916,504.8  poll                  
      8.4      165,577,820        525     315,3